# 05 · 실제 출장 도우미 챗봇 조립

## 학습 목표
- 사규 검색·날씨·환율을 단일 에이전트에 연결합니다.
- 후속 질문, 세션 초기화, 출처와 도구 실행 기록을 확인합니다.
- Notebook 실험 코드를 Streamlit 앱으로 연결합니다.

## 개념
`ChatSession`은 한 사용자 전용입니다. 검색은 `search_policy`, 외부 정보는 `get_weather`/`convert_currency`가 담당합니다. 도구 실행 기록은 실제 함수 실행의 이름·성공 여부·소요 시간이며 모델의 숨겨진 추론 과정이 아닙니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


In [ ]:
from hanbit_assistant.agent import ChatSession
chat = ChatSession(settings)
try:
    questions = [
        "도쿄 출장의 1인 1박 숙박비 한도를 알려줘.",
        "그 한도를 원화로 환산해줘. 사규의 원래 금액을 다시 확인하고 참고 환율 기준일도 알려줘.",
        "도쿄의 현재 날씨도 알려줘.",
    ]
    for question in questions:
        turn = await chat.ask(question)
        print("질문:", question)
        print(turn.text)
        print("실제 호출:", turn.tool_events)
        print("검색 출처:", [(s["source"], s["section"]) for s in turn.sources])
    chat.reset()
    print("초기화 후 대화 횟수:", chat.turn_count)
    assert chat.turn_count == 0
finally:
    await chat.close()


### VS Code 터미널에서 챗봇 실행
```bash
uv run streamlit run app/streamlit_app.py
```
브라우저를 열어 다음을 확인하세요.
- 질문과 답변, 검색 근거 펼치기, 실제 도구 실행 내역
- “새 대화 / 초기화” 후 이전 사용자 정보가 없어짐
- 새 브라우저 세션과 기존 대화가 분리됨

앱은 인덱스를 자동으로 생성하지 않습니다. 03의 명시적인 쓰기 실습과 실제 조회 앱을 분리했습니다. 앱을 공용 인터넷 서비스로 배포하는 것은 이 실습 범위가 아닙니다.

## 관찰
사규의 원래 통화 금액과 현재 참고 환산액을 구분해야 합니다. 현재 환율로 환산했다고 회사 정산 기준이 바뀌는 것은 아닙니다. 원격 응답 저장은 꺼져 있지만 대화는 현재 Python/브라우저 세션 메모리에 남습니다.

## 연습
“호텔을 예약하고 회사 카드로 결제해줘”라고 요청하면 무엇을 해야 할까요? 실제 회사 API를 추가한다면 어떤 안전장치가 더 필요할까요?

<details><summary>예시 해설</summary>현재 도구는 모두 읽기 전용이라 예약·결제를 수행할 수 없다고 명확히 답합니다. 쓰기 도구에는 인증된 사용자 권한, 명시적 확인, 중복 방지, 감사 기록, 실패 복구가 추가로 필요합니다. 선택 확장으로 검색 담당과 최종 검토 담당을 나눌 수 있지만 이번 기본 과정은 단일 에이전트만으로 충분합니다.</details>